In [2]:
import ee
ee.Initialize()

/home/wmlegion/miniconda3/envs/agri_land_env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    # Initialize Nominatim geocoder (requires a custom user_agent string)
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")

    try:
        # Perform reverse geocoding
        location = geolocator.reverse((lat, lon), language='en')

        if location and location.raw:
            address = location.raw.get('address', {})

            # Extract city/town/village and country safely
            city = (
                address.get('city') or 
                address.get('town') or 
                address.get('village') or 
                address.get('municipality') or 
                address.get('county') or 
                "Unknown City"
            )
            country = address.get('country', "Unknown Country")

            return {"city": city, "country": country}
        else:
            return {"city": "Not found", "country": "Not found"}

    except Exception as e:
        return {"error": str(e)}

# Test it with your coordinates (7.3297, -73.1867)
location_info = get_location_name(7.300921,-73.009794)
print("Location:", location_info)
#Location: {'city': 'Bundaberg', 'country': 'Australia'}(-24.8660, 152.3489)

Location: {'city': 'Matanza', 'country': 'Colombia'}


In [ ]:
from geopy.geocoders import Nominatim

def get_location_name(lat, lon):
    # Initialize Nominatim geocoder (requires a custom user_agent string)
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")

    try:
        # Perform reverse geocoding
        location = geolocator.reverse((lat, lon), language='en')

        if location and location.raw:
            address = location.raw.get('address', {})

            # Extract city/town/village safely
            city = (
                address.get('city') or 
                address.get('town') or 
                address.get('village') or 
                address.get('municipality') or 
                address.get('county') or 
                "Unknown City"
            )
            
            # Extract state/province safely (handles different regional naming conventions)
            state = (
                address.get('state') or 
                address.get('province') or 
                address.get('region') or 
                address.get('state_district') or 
                "Unknown State"
            )
            
            country = address.get('country', "Unknown Country")

            return {"city": city, "state": state, "country": country}
        else:
            return {"city": "Not found", "state": "Not found", "country": "Not found"}

    except Exception as e:
        return {"error": str(e)}

# Test with your coordinates where city came up as Unknown City
# Finca Matanza 7.300921,-73.009794
location_info = get_location_name(-24.8660, 152.3489)
print("Location:", location_info)

Location: {'city': 'Bundaberg', 'state': 'Queensland', 'country': 'Australia'}


In [7]:
print(location_info['state'])

Queensland


In [10]:
import math
import time
import requests
from PIL import Image
from geopy.geocoders import Nominatim

# --- 1. Obtener el bounding box del state ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError("No se encontró el state")
    # boundingbox viene como [south, north, west, east] en strings
    south, north, west, east = map(float, location.raw['boundingbox'])
    return south, north, west, east

# --- 2. Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y

def tile_to_latlon(x, y, zoom):
    n = 2 ** zoom
    lon = x / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y / n)))
    lat = math.degrees(lat_rad)
    return lat, lon

# --- 3. Descargar y pegar los tiles ---
def get_state_elevation_map(state_name, country="Colombia", zoom=10, out_path="state_map.png"):
    south, north, west, east = get_state_bbox(state_name, country)

    x_min, y_min = latlon_to_tile(north, west, zoom)  # esquina superior-izq
    x_max, y_max = latlon_to_tile(south, east, zoom)  # esquina inferior-der

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code == 200:
                tile_img = Image.open(requests.compat.io.BytesIO(resp.content)) if False else None
            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)
            try:
                from io import BytesIO
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path

# Prueba
get_state_elevation_map("Santander", "Colombia", zoom=9)


Mapa guardado en state_map.png (768x1024px)


'state_map.png'

In [ ]:
import math
import time
from io import BytesIO
import requests
from PIL import Image
from geopy.geocoders import Nominatim


# --- 1. Obtener el bounding box del state ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError(f"No se encontró el state '{state_name}, {country}' en Nominatim")
    print(f"[DEBUG] Nominatim encontró: {location.address}")
    south, north, west, east = map(float, location.raw['boundingbox'])
    print(f"[DEBUG] bbox -> south={south}, north={north}, west={west}, east={east}")
    return south, north, west, east


# --- 2. Conversión lat/lon <-> número de tile ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


# --- 3. Descargar y pegar los tiles (con debug) ---
def get_state_elevation_map(state_name, country="Colombia", zoom=9, out_path="state_map.png"):
    south, north, west, east = get_state_bbox(state_name, country)

    x_min, y_min = latlon_to_tile(north, west, zoom)
    x_max, y_max = latlon_to_tile(south, east, zoom)
    print(f"[DEBUG] tiles a descargar: x {x_min}-{x_max}, y {y_min}-{y_max} "
          f"({(x_max - x_min + 1) * (y_max - y_min + 1)} tiles)")

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    # OJO: pon un email/URL real tuyo aquí, no un placeholder falso.
    # Muchos servidores de tiles rechazan (403) User-Agents genéricos.
    headers = {"User-Agent": "agri_land_suitability_pipeline/1.0 (contacto: tu_correo_real@dominio.com)"}

    exitosos, fallidos = 0, 0
    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            print(f"[DEBUG] tile {x},{y} -> status {resp.status_code}, "
                  f"{len(resp.content)} bytes, content-type={resp.headers.get('Content-Type')}")

            if resp.status_code != 200:
                print(f"[DEBUG]   contenido de la respuesta (primeros 200 chars): {resp.text[:200]}")
                fallidos += 1
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
                exitosos += 1
            except Exception as e:
                print(f"[DEBUG]   Image.open falló para tile {x},{y}: {e}")
                fallidos += 1

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    print(f"[DEBUG] tiles exitosos: {exitosos}, fallidos: {fallidos}")
    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Reemplaza esto con los valores reales de tu location_info
    # location_info = {"state": "Santander", "country": "Colombia"}
    get_state_elevation_map(location_info['state'], location_info['country'], zoom=9)

[DEBUG] Nominatim encontró: Queensland, Australia
[DEBUG] bbox -> south=-29.179266, north=-9.0880125, west=137.9946464, east=153.6116035
[DEBUG] tiles a descargar: x 452-474, y 268-299 (736 tiles)
[DEBUG] tile 452,268 -> status 200, 3290 bytes, content-type=image/png
[DEBUG] tile 452,269 -> status 200, 629 bytes, content-type=image/png
[DEBUG] tile 452,270 -> status 200, 756 bytes, content-type=image/png
[DEBUG] tile 452,271 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,272 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,273 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,274 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,275 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,276 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,277 -> status 200, 103 bytes, content-type=image/png
[DEBUG] tile 452,278 -> status 200, 1415 bytes, content-type=image/png
[DEBUG] tile 452,279 -> status 

In [13]:
print(location_info['state'])
print(location_info['country'])

Queensland
Australia


In [5]:
import os
import elevation
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from geopy.geocoders import Nominatim


# --- 1. Obtener el bounding box del state (igual que antes) ---
def get_state_bbox(state_name, country="Colombia"):
    geolocator = Nominatim(user_agent="agri_land_suitability_pipeline")
    location = geolocator.geocode(f"{state_name}, {country}", exactly_one=True)
    if not location:
        raise ValueError(f"No se encontró el state '{state_name}, {country}' en Nominatim")
    south, north, west, east = map(float, location.raw['boundingbox'])
    print(f"[DEBUG] bbox -> south={south}, north={north}, west={west}, east={east}")
    return south, north, west, east


# --- 2. Descargar el DEM recortado al bbox (un solo GeoTIFF) ---
def get_state_dem(state_name, country="Colombia", out_path="state_dem.tif"):
    south, north, west, east = get_state_bbox(state_name, country)

    # elevation.clip espera (west, south, east, north)
    bounds = (west, south, east, north)
    print(f"[DEBUG] descargando DEM para bounds={bounds} ...")

    # max_download_tiles: la librería limita por defecto la cantidad de tiles SRTM
    # para evitar bulk-download accidental. Lo subimos a un tope razonable para
    # cubrir states grandes (Queensland necesitaría bastantes más que Santander).
    elevation.clip(bounds=bounds, output=os.path.abspath(out_path), max_download_tiles=50)
    elevation.clean()  # borra los tiles temporales intermedios, deja solo el resultado

    print(f"[DEBUG] DEM guardado en {out_path}")
    return out_path


# --- 3. Renderizar el DEM con hillshade (visual similar a OpenTopoMap) ---
def render_dem(dem_path, out_image="state_map.png"):
    with rasterio.open(dem_path) as src:
        elevation_data = src.read(1).astype(float)
        elevation_data[elevation_data == src.nodata] = np.nan

    ls = LightSource(azdeg=315, altdeg=45)
    rgb = ls.shade(elevation_data, cmap=plt.cm.terrain, vert_exag=1, blend_mode='soft')

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(rgb)
    ax.axis('off')
    plt.savefig(out_image, dpi=200, bbox_inches='tight', pad_inches=0)
    plt.close()
    print(f"[DEBUG] Imagen renderizada en {out_image}")
    return out_image


if __name__ == "__main__":
    # Reemplaza con location_info real de tu pipeline
    location_info = {"state": "Santander", "country": "Colombia"}

    dem_path = get_state_dem(location_info['state'], location_info['country'])
    render_dem(dem_path)

[DEBUG] bbox -> south=5.7067098, north=8.1434194, west=-74.5266563, east=-72.4764662
[DEBUG] descargando DEM para bounds=(-74.5266563, 5.7067098, -72.4764662, 8.1434194) ...
make: Entering directory '/home/wmlegion/.cache/elevation/SRTM1'
curl -s -o spool/N05/N05W075.hgt.gz.temp https://s3.amazonaws.com/elevation-tiles-prod/skadi/N05/N05W075.hgt.gz && mv spool/N05/N05W075.hgt.gz.temp spool/N05/N05W075.hgt.gz
gunzip spool/N05/N05W075.hgt.gz 2>/dev/null || touch spool/N05/N05W075.hgt
gdal_translate -q -co TILED=YES -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2 spool/N05/N05W075.hgt cache/N05/N05W075.tif 2>/dev/null || touch cache/N05/N05W075.tif
curl -s -o spool/N06/N06W075.hgt.gz.temp https://s3.amazonaws.com/elevation-tiles-prod/skadi/N06/N06W075.hgt.gz && mv spool/N06/N06W075.hgt.gz.temp spool/N06/N06W075.hgt.gz
gunzip spool/N06/N06W075.hgt.gz 2>/dev/null || touch spool/N06/N06W075.hgt
gdal_translate -q -co TILED=YES -co COMPRESS=DEFLATE -co ZLEVEL=9 -co PREDICTOR=2 spool/N06/N06W

In [12]:
import math
import time
from io import BytesIO
import requests
from PIL import Image


def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def get_point_map(lat, lon, zoom=17, radius=2, out_path="point_map.png"):
    """
    Descarga un recorte cuadrado de tiles centrado en (lat, lon).
    radius=2 -> grilla de (2*2+1)x(2*2+1) = 5x5 tiles alrededor del centro.
    Sube 'radius' si quieres más contexto alrededor del punto.
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    n_tiles = (x_max - x_min + 1) * (y_max - y_min + 1)
    print(f"[DEBUG] centro del tile: {x_center},{y_center} (zoom {zoom})")
    print(f"[DEBUG] descargando {n_tiles} tiles (grilla {2*radius+1}x{2*radius+1})")

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline/1.0 (contacto: tu_correo_real@dominio.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code != 200:
                print(f"[DEBUG] tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue
            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"[DEBUG] tile {x},{y} no se pudo abrir: {e}")
            time.sleep(0.5)

    canvas.save(out_path)
    print(f"[DEBUG] Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Punto de ejemplo que diste
    get_point_map(7.3297, -73.1867, zoom=17, radius=2)

[DEBUG] centro del tile: 38889,62860 (zoom 17)
[DEBUG] descargando 25 tiles (grilla 5x5)
[DEBUG] Mapa guardado en point_map.png (1280x1280px)


In [ ]:
import math
import time
from io import BytesIO
import requests
from PIL import Image


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_    _map(lat, lon, zoom=17, radius=2, out_path="point_map.png"):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(7.300921,-73.009794, zoom=17, radius=2)

Mapa guardado en point_map.png (1280x1280px)


'point_map.png'

In [14]:
import math
import time
from io import BytesIO
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado.
    Esto es lo que permite ubicar el punto exacto dentro del canvas, no solo el tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_point_map(lat, lon, zoom=17, radius=2, out_path="point_map.png", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    if show_marker:
        # Pixel absoluto del punto en el mundo, menos el origen del canvas (esquina x_min,y_min)
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(7.300921,-73.009794, zoom=17, radius=2)

Mapa guardado en point_map.png (1280x1280px)


'point_map.png'

In [3]:
import math
import time
from datetime import datetime
from io import BytesIO
import requests
from PIL import Image, ImageDraw


# --- Conversión lat/lon <-> número de tile (fórmula estándar Web Mercator) ---
def latlon_to_tile(lat, lon, zoom):
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def latlon_to_pixel(lat, lon, zoom, tile_size=256):
    """Convierte lat/lon a coordenadas de PIXEL absolutas (no solo tile) en el zoom dado.
    Esto es lo que permite ubicar el punto exacto dentro del canvas, no solo el tile."""
    lat_rad = math.radians(lat)
    n = 2 ** zoom
    x_pixel = (lon + 180.0) / 360.0 * n * tile_size
    y_pixel = (1.0 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2.0 * n * tile_size
    return x_pixel, y_pixel


def draw_marker(canvas, x, y, radius=8, color=(220, 30, 30)):
    """Dibuja un marcador circular con contorno blanco en la posición (x, y) del canvas."""
    draw = ImageDraw.Draw(canvas)
    draw.ellipse(
        [(x - radius, y - radius), (x + radius, y + radius)],
        fill=color, outline=(255, 255, 255), width=3
    )


# --- Descargar y pegar los tiles alrededor de un punto ---
def get_point_map(lat, lon, zoom=17, radius=2, out_prefix="point_map", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    zoom: nivel de zoom (OpenTopoMap soporta hasta 17)
    radius: cuántos tiles agregar alrededor del centro en cada dirección.
            radius=2 -> grilla de 5x5 tiles
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    out_prefix: nombre base del archivo; se le agrega -vYYMMDDHHMMSS.png automáticamente
    """
    # Timestamp al momento de generar el mapa: YYMMDDHHMMSS (ej: 260803143022)
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    out_path = f"{out_prefix}-v{timestamp}.png"

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email@ejemplo.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = f"https://tile.opentopomap.org/{zoom}/{x}/{y}.png"
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(0.5)
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(0.5)  # respeta el rate limit (~2 req/seg máx)

    if show_marker:
        # Pixel absoluto del punto en el mundo, menos el origen del canvas (esquina x_min,y_min)
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Mapa guardado en {out_path} ({width}x{height}px)")
    return out_path


# Prueba
get_point_map(3.580109040361371, -76.31299479308868, zoom=17, radius=2)

Mapa guardado en point_map-v260803164014.png (1280x1280px)


'point_map-v260803164014.png'